In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking (looks like this doesn't work in the current version of the API, but we'll keep it here for future use)
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
messages = []

add_user_message(messages, thinking_test_str)

chat(messages, thinking=True)

Message(id='msg_01GbAntCSrhpjNgt8x4TnLdA', container=None, content=[ThinkingBlock(signature='EsgFCm4IDhgCKkDXBsEXoi1MXVft6XCxExi80I+85M5NOls6KFpRcmuzviObc2gNyN3oLDhF/PQMr293SusCtwubPOnjGKzm+FAnMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAQgh0aGlua2luZxIMjqZeLpQ3AiRHUCFZGgwSMWnIlS7uzbL8lmYiMPshTQAq7beq0Ri5mzEYtxEHl8t+YHBglXiLiavjGdVA1EV3t24Pm8AU9tEXRbOIeCqHBJYwZw28booN9rSy6s9i83ErIvIJkBvq/Srg/j5rn2AlHvZ/bjSXBkmupYV0L8VabJDRLQklCdgRtUdrUkb5scInhqXN9htGGq+a5jCRISknkz1GIHL6WBVBriB+JL1BPuXdkRKJ9QtGIq9PCDe6TSOGNA+/4lwpp+67yIz/5+5xjeEwuGowjC8wtXERylyKoj/c7dUlgzRyldMnXIQrmG0o87LOjG6rTQR5KIKm+MhPcA4Xz78EGTd8FZM0EJm7p3m5muYpUt4frZnFYvii0RAFjWBRh6LA8iPFiUEodivBjdxrI6u6iNmYW4Skdk0d8NDJuyBD2+1byTbrdtTyDdOoMe8hHdlvwqxdTkGLNTwDJsgtlt0L/hMDjgUUpO97aPFIgoR7VF+CtwiZY2JtAu/ux9HbeTnukSYNjOyD1R+wTylo7Q8F0KWRQhbnLvmVrrWTMmR+csYHAk15EurADXZQVfqFalbDcjBY7+bkLtlXltJrIK20kzQ/18fnvrKR90qReNKzHKvh1WIiFPBghXKpsbZZ63eZdwlMXYJDbQ/7IcF+jrbJoAKvF4CqtoYpI5sLxjek6rtQaBauCc+n+nmREaZkiKmghqxic+oCZ4ifSictbDGSTZqyoV2nFiAKG1CEkO8